# 📦 RMUC 消息拆分说明文档

> **项目**: RoboMaster RMUC 2026 哨兵行为树  
> **日期**: 2026-02-25  
> **分支**: `test`

---

### 背景

原始设计中，所有 RMUC 比赛数据封装在一个 **272 行的 `RMUC.msg`** 大消息中，通过统一的 `/rmuc` 话题传输。这带来了三个问题：

| 问题 | 影响 |
|:---|:---|
| **频率耦合** | 1 Hz 的比赛状态和 50 Hz 的位姿数据被迫以相同频率发布 |
| **带宽浪费** | 每帧都传输完整消息，即使只有少数字段更新 |
| **调试困难** | `ros2 topic echo` 输出刷屏，难以定位单一数据源问题 |

**改进方案**：将 `RMUC.msg` 拆分为 **8 个独立的小消息**，各自拥有独立话题、独立频率，实现数据解耦。

```
原架构:  [所有数据] ──→ /rmuc (单话题)
新架构:  [比赛状态] ──→ /game_status        (1 Hz)
         [机器人状态] ──→ /robot_status       (10 Hz)
         [RFID]      ──→ /rfid_status        (事件驱动)
         [位姿]      ──→ /robot_position     (50 Hz)
         [雷达]      ──→ /radar/enemy_tracks (10-30 Hz)
         [决策指令]  ←── /sentry_cmd         (2 Hz)
         [云台控制]  ←── /robot_control      (10 Hz)
         [导航控制]  ←── /nav_control_cmd    (按需)
```

## 1. 拆分后的 8 个消息详情

> 所有 `.msg` 文件位于 `rm_decision_interfaces/msg/RMUC/` 目录下。  
> 箭头方向：`→` 表示 BT 订阅（输入），`←` 表示 BT 发布（输出）。

---

### 1.1 📥 RMUCGameStatus — 比赛状态
| 属性 | 值 |
|:---|:---|
| **话题** | `/game_status` |
| **频率** | 1 Hz |
| **方向** | 裁判系统 → BT |

| 字段 | 类型 | 说明 |
|:---|:---|:---|
| `game_progress` | `uint8` | 比赛阶段 (0=未开始, 4=进行中, 5=结算) |
| `stage_remain_time` | `uint16` | 当前阶段剩余时间 (秒) |

---

### 1.2 📥 RMUCRobotStatus — 机器人核心状态
| 属性 | 值 |
|:---|:---|
| **话题** | `/robot_status` |
| **频率** | 10 Hz |
| **方向** | 电控 → BT |

| 字段分组 | 包含字段 |
|:---|:---|
| 血量与热量 | `current_hp`, `max_hp`, `shooter_heat`, `heat_limit`, `cooling_rate` |
| 弹丸 | `ammo_allow`, `ammo_left` |
| 状态标志 | `is_dead`, `is_weak`, `is_disengaged`, `disengage_cd_s` |
| 经济能力 | `can_remote_heal`, `can_remote_ammo`, `team_coins` |
| 复活 | `can_respawn`, `respawn_countdown_s` |
| 基地与前哨 | `base_hp_cur`, `base_hp_max`, `outpost_alive` |

---

### 1.3 📥 RMUCRFIDStatus — RFID 场地交互
| 属性 | 值 |
|:---|:---|
| **话题** | `/rfid_status` |
| **频率** | 事件驱动 (变化时发布) |
| **方向** | 电控 → BT |

7 个 `bool` 字段：`rfid_supply`, `rfid_base_buff`, `rfid_outpost_buff`, `rfid_fortress_ally`, `rfid_fortress_enemy`, `rfid_central_highland`, `rfid_ladder_highland`

---

### 1.4 📥 RMUCRobotPosition — 自身位姿
| 属性 | 值 |
|:---|:---|
| **话题** | `/robot_position` |
| **频率** | 50 Hz |
| **方向** | 定位系统 → BT |

| 字段 | 类型 | 说明 |
|:---|:---|:---|
| `pose_x` | `float32` | 地图坐标系 x (m) |
| `pose_y` | `float32` | 地图坐标系 y (m) |
| `pose_yaw` | `float32` | 航向角 (rad) |
| `is_at_nav_goal` | `bool` | 是否已到达导航目标点 |

---

### 1.5 📥 RMUCEnemyTracks — 雷达敌方跟踪
| 属性 | 值 |
|:---|:---|
| **话题** | `/radar/enemy_tracks` |
| **频率** | 10-30 Hz |
| **方向** | 雷达站 → BT |

| 字段 | 类型 | 说明 |
|:---|:---|:---|
| `enemy_count` | `uint8` | 有效跟踪数 (0~7) |
| `enemy_robot_id[]` | `uint8[]` | 敌方机器人 ID |
| `enemy_x[]` / `enemy_y[]` | `float32[]` | 地图坐标位置 |
| `enemy_confidence[]` | `float32[]` | 置信度 [0, 1] |

---

### 1.6 📤 RMUCSentryCmd — 哨兵决策指令
| 属性 | 值 |
|:---|:---|
| **话题** | `/sentry_cmd` |
| **频率** | 2 Hz |
| **方向** | BT → 电控 |

| 字段 | 说明 |
|:---|:---|
| `cmd_posture` | 姿态指令 (1=进攻, 2=防御, 3=移动) |
| `cmd_confirm_respawn` | 确认复活 |
| `cmd_confirm_instant_respawn` | 确认立即复活 |
| `cmd_allow_ammo_target` | 允许发弹量目标 |
| `cmd_trigger_remote_ammo` | 远程补弹 (上升沿) |
| `cmd_trigger_remote_hp` | 远程补血 (上升沿) |
| `cmd_enable_big_energy` | 大能量机关确认 |

---

### 1.7 📤 RMUCRobotControl — 云台与底盘控制
| 属性 | 值 |
|:---|:---|
| **话题** | `/robot_control` |
| **频率** | 10 Hz |
| **方向** | BT → 电控 |

| 字段 | 类型 | 说明 |
|:---|:---|:---|
| `stop_gimbal_scan` | `bool` | 停止云台扫描 |
| `chassis_spin` | `bool` | 底盘小陀螺 |
| `fire_enable` | `bool` | 允许发射 |

---

### 1.8 📤 RMUCNavControlCmd — 导航控制
| 属性 | 值 |
|:---|:---|
| **话题** | `/nav_control_cmd` |
| **频率** | 按需发布 |
| **方向** | BT → 电控 |

| 字段 | 类型 | 说明 |
|:---|:---|:---|
| `cmd_type` | `int32` | 0=无操作, 1=开始, 2=终止, 3=原地 |
| `emergency_stop` | `bool` | 紧急停止 |

## 2. 话题映射表 (旧 → 新)

| 方向 | 原 `/rmuc` 字段类别 | 新消息类型 | 独立话题 | BT 插件 | 频率 |
|:---:|:---|:---|:---|:---|:---:|
| 📥 | Game Status | `RMUCGameStatus` | `/game_status` | `RmucSubGameStatus` | 1 Hz |
| 📥 | Robot HP / Heat / Ammo | `RMUCRobotStatus` | `/robot_status` | `RmucSubRobotStatus` | 10 Hz |
| 📥 | RFID Flags | `RMUCRFIDStatus` | `/rfid_status` | `RmucSubRFIDStatus` | 事件 |
| 📥 | Robot Pose | `RMUCRobotPosition` | `/robot_position` | `RmucSubRobotPosition` | 50 Hz |
| 📥 | Radar Tracks | `RMUCEnemyTracks` | `/radar/enemy_tracks` | `SubRadarTracks` | 10-30 Hz |
| 📤 | Sentry Decision | `RMUCSentryCmd` | `/sentry_cmd` | `SentryCmdMux` | 2 Hz |
| 📤 | Gimbal / Chassis | `RMUCRobotControl` | `/robot_control` | `RmucRobotControl` | 10 Hz |
| 📤 | Nav Control | `RMUCNavControlCmd` | `/nav_control_cmd` | `RmucNavControlCmd` | 按需 |

## 3. 插件引用关系

> 📌 **图例**：🔵 订阅者 · 🟢 条件节点 · 🟠 动作节点 · 🔴 解析/混合

---

### `RMUCGameStatus` → 3 个插件
| 插件 | 类型 | 说明 |
|:---|:---:|:---|
| `RmucSubGameStatus` | 🔵 | 订阅 `/game_status`，写入黑板 `{game_status}` |
| `IsGameTime` | 🟢 | 判断当前比赛阶段是否为指定值 |
| `ParseSentryBlackboard` | 🔴 | 解析 `stage_remain_time` 等派生变量 |

### `RMUCRobotStatus` → 9 个插件 ⭐ 引用最多
| 插件 | 类型 | 说明 |
|:---|:---:|:---|
| `RmucSubRobotStatus` | 🔵 | 订阅 `/robot_status`，写入黑板 `{robot_status}` |
| `DetectRespawnAndSetRecovery` | 🔴 | 订阅 + 检测复活沿 |
| `WaitAndHeal` | 🔴 | 订阅 + 等待回血到指定比例 |
| `IsDead` | 🟢 | 判断 `is_dead` |
| `IsHPBelow` | 🟢 | 判断 `current_hp < threshold` |
| `DecideRespawnCmd` | 🟠 | 读取复活状态生成决策指令 |
| `ParseSentryBlackboard` | 🔴 | 解析 HP/弹丸/经济等派生变量 |
| `IsZoneCardDetected` | 🟢 | 混合引用 (同时读 RFID) |
| `IsAnyDispelCardDetected` | 🟢 | 混合引用 (同时读 RFID) |

### `RMUCRFIDStatus` → 5 个插件
| 插件 | 类型 | 说明 |
|:---|:---:|:---|
| `RmucSubRFIDStatus` | 🔵 | 订阅 `/rfid_status`，写入黑板 `{rfid.status}` |
| `IsSupplyCardDetected` | 🟢 | 判断 `rfid_supply` |
| `IsZoneCardDetected` | 🟢 | 判断多个增益区 RFID |
| `IsAnyDispelCardDetected` | 🟢 | 判断是否触发任意驱散点 |
| `MicroSearchSupplyCard` | 🟠 | 微调搜索补给卡 |

### `RMUCRobotPosition` → 2 个插件
| 插件 | 类型 | 说明 |
|:---|:---:|:---|
| `RmucSubRobotPosition` | 🔵 | 订阅 `/robot_position`，输出 `{pose.x/y/yaw}` + `{is_at_nav_goal}` |
| `IsAtNavGoal` | 🟢 | 读取 `bool is_at_nav_goal` 黑板变量 |

### `RMUCEnemyTracks` → 3 个插件
| 插件 | 类型 | 说明 |
|:---|:---:|:---|
| `SubRadarTracks` | 🔵 | 订阅 `/radar/enemy_tracks`，写入黑板 `{radar.tracks}` |
| `SelectBestTarget` | 🟠 | 根据距离/置信度选择最优打击目标 |
| `ParseSentryBlackboard` | 🔴 | 解析 `has_target` / `best_target` 等 |

## 4. 主程序 RosNodeParams 配置

> 📄 文件：`rm_behavior_tree/rm_behavior_tree/src/rm_behavior_tree_rmuc.cpp`

改造前，所有 RMUC 订阅/发布插件共用一个 `params_rmuc`（绑定 `/rmuc` 话题）。  
改造后，**每个话题分配独立的 `RosNodeParams`**，各自拥有独立的 ROS 节点和默认话题名。

---

### 4.1 参数定义一览

```cpp
// ── 5 个输入订阅参数 ──
BT::RosNodeParams params_game_status;
params_game_status.nh = std::make_shared<rclcpp::Node>("rmuc_game_status_io");
params_game_status.default_port_value = "/game_status";

BT::RosNodeParams params_robot_status;
params_robot_status.nh = std::make_shared<rclcpp::Node>("rmuc_robot_status_io");
params_robot_status.default_port_value = "/robot_status";

BT::RosNodeParams params_rfid_status;
params_rfid_status.nh = std::make_shared<rclcpp::Node>("rmuc_rfid_status_io");
params_rfid_status.default_port_value = "/rfid_status";

BT::RosNodeParams params_robot_position;
params_robot_position.nh = std::make_shared<rclcpp::Node>("rmuc_robot_position_io");
params_robot_position.default_port_value = "/robot_position";

BT::RosNodeParams params_radar;
params_radar.nh = std::make_shared<rclcpp::Node>("rmuc_radar_io");
params_radar.default_port_value = "/radar/enemy_tracks";

// ── 3 个输出发布参数 ──
BT::RosNodeParams params_sentry_cmd;
params_sentry_cmd.nh = std::make_shared<rclcpp::Node>("rmuc_sentry_cmd_io");
params_sentry_cmd.default_port_value = "/sentry_cmd";

BT::RosNodeParams params_robot_ctrl;
params_robot_ctrl.nh = std::make_shared<rclcpp::Node>("rmuc_robot_ctrl_io");
params_robot_ctrl.default_port_value = "/robot_control";

BT::RosNodeParams params_nav_cmd;
params_nav_cmd.nh = std::make_shared<rclcpp::Node>("rmuc_nav_cmd_io");
params_nav_cmd.default_port_value = "/nav_control_cmd";
```

### 4.2 插件注册分组

```cpp
// ── Game Status 组 ──
factory.registerNodeType<RmucSubGameStatus>("SubGameStatus", params_game_status);

// ── Robot Status 组 ──
factory.registerNodeType<RmucSubRobotStatus>("SubRobotStatus", params_robot_status);
factory.registerNodeType<DetectRespawnAndSetRecovery>("DetectRespawnAndSetRecovery", params_robot_status);
factory.registerNodeType<WaitAndHeal>("WaitAndHeal", params_robot_status);

// ── RFID Status 组 ──
factory.registerNodeType<RmucSubRFIDStatus>("SubRFIDStatus", params_rfid_status);
factory.registerNodeType<MicroSearchSupplyCard>("MicroSearchSupplyCard", params_rfid_status);
factory.registerNodeType<IsSupplyCardDetected>("IsSupplyCardDetected", params_rfid_status);

// ── Robot Position 组 ──
factory.registerNodeType<RmucSubRobotPosition>("SubRobotPosition", params_robot_position);

// ── Radar 组 ──
factory.registerNodeType<SubRadarTracks>("SubRadarTracks", params_radar);

// ── 输出组 ──
factory.registerNodeType<SentryCmdMux>("SentryCmdMux", params_sentry_cmd);
factory.registerNodeType<RmucRobotControl>("RmucRobotControl", params_robot_ctrl);
factory.registerNodeType<RmucNavControlCmd>("RmucNavControlCmd", params_nav_cmd);
```

### 4.3 XML 端话题覆盖

虽然 `default_port_value` 已设置默认话题，XML 中仍可通过 `topic_name` 属性覆盖。  
以下列出全部 **8 个话题**（5 📥 订阅 + 3 📤 发布）的 XML 配置：

```xml
<!-- ═══ 📥 5 个输入订阅 (PerceptionAndBlackboard.xml) ═══ -->
<SubGameStatus    topic_name="/game_status"        game_status="{game_status}" />
<SubRobotStatus   topic_name="/robot_status"       robot_status="{robot_status}" />
<SubRFIDStatus    topic_name="/rfid_status"         rfid_status="{rfid.status}" />
<SubRobotPosition topic_name="/robot_position"      ... is_at_nav_goal="{nav.at_goal}" />
<SubRadarTracks   topic_name="/radar/enemy_tracks"  radar_tracks="{radar.tracks}" />

<!-- ═══ 📤 3 个输出发布 (分布在不同子树中) ═══ -->
<!-- CommandHub.xml -->
<SentryCmdMux     topic_name="/sentry_cmd"         ... />
<!-- 多个战术子树 (DeathAndRespawn / EngageCombat / PatrolAndScan 等) -->
<RmucRobotControl topic_name="/robot_control"      stop_gimbal_scan="..." chassis_spin="..." fire_enable="..." />
<!-- RespawnRecovery.xml -->
<RmucNavControlCmd topic_name="/nav_control_cmd"   cmd_type="..." emergency_stop="..." />
```

## 5. 修改记录与踩坑备忘

### 📝 修改文件清单

| 类别 | 文件 / 路径 | 修改内容 |
|:---|:---|:---|
| **消息定义** | `rm_decision_interfaces/msg/RMUC/*.msg` (×8) | 新增 8 个拆分消息 |
| **接口构建** | `rm_decision_interfaces/CMakeLists.txt` | 添加 8 个 msg 到 `rosidl_generate_interfaces` |
| **插件头文件** | `rmuc_plugins/*.hpp` (×21) | include 路径 + 类型名替换 |
| **插件源文件** | `rmuc_plugins/*.cpp` (×21) | 类型名替换 |
| **主程序入口** | `rm_behavior_tree_rmuc.cpp` | 重写为 8 组独立 RosNodeParams |
| **XML 子树** | `PerceptionAndBlackboard.xml` | 5 个订阅者话题 + 新增 is_at_nav_goal 输出 |
| **XML 子树** | `RespawnRecovery.xml` | topic_name 更新 + IsAtNavGoal 改为读 bool |
| **XML 主树** | `rmuc_2026.xml` | TreeNodesModel 默认值全部更新 |
| **测试脚本** | `test_rmuc_bt.py` | 改为 5 话题独立发布 |

---

### ⚠️ 踩坑记录

#### 1. 生成头文件路径是扁平的
```
❌ #include "rm_decision_interfaces/msg/rmuc/rmuc_game_status.hpp"
✅ #include "rm_decision_interfaces/msg/rmuc_game_status.hpp"
```
> `rosidl` 生成的头文件统一放在 `msg/` 目录下，不会保留源文件的子目录结构。

#### 2. `RMUCRFIDStatus` 的 snake_case 没有下划线
```
❌ rmuc_rfid_status.hpp
✅ rmucrfid_status.hpp
```
> ROS 2 的 CamelCase → snake_case 规则：`RMUC` → `rmuc`，`RFID` → `rfid`，中间**不插入**下划线。可用 `ros2 interface show` 验证。

#### 3. sed 批量替换导致双后缀
```
❌ RMUCGameStatusGameStatus  (已含后缀的文件被二次替换)
✅ RMUCGameStatus
```
> 对已完成部分替换的文件执行全局 `RMUC` → `RMUCGameStatus`，导致 `RMUCGameStatus` 被再次替换。解决：先检查是否已替换。

#### 4. `IsAtNavGoal` 类型冲突
```
原: InputPort<RMUCRobotPosition>("rfid_status")  ← 类型+key 均不匹配
改: InputPort<bool>("is_at_nav_goal")             ← 从 SubRobotPosition 输出的 bool
```
> `SubRobotPosition` 新增 `is_at_nav_goal` 输出端口，写入黑板 `{nav.at_goal}`；`IsAtNavGoal` 直接读取该 bool 值。

#### 5. XML `topic_name` 优先于 `default_port_value`
> 即使 C++ 中设置了 `params.default_port_value = "/robot_status"`，XML 中若残留 `topic_name="/rmuc"` 会覆盖默认值。**必须同步修改 XML**。

---

### ✅ 验证方式

```bash
# 1. 编译
colcon build --packages-select rm_decision_interfaces rm_behavior_tree

# 2. 启动行为树
ros2 run rm_behavior_tree rm_behavior_tree_rmuc \
  --ros-args -p style:=<path_to>/rmuc_2026.xml

# 3. 另一终端运行测试脚本
python3 test_rmuc_bt.py

# 4. 检查话题列表
ros2 topic list | grep -E "game_status|robot_status|rfid_status|robot_position|enemy_tracks"
```